## Problem Statement

### Business Context

GlobalEdge Brokerage is a mid-sized brokerage firm operating across multiple countries, serving thousands of retail and institutional clients through a network of equity brokers. Brokers handle investment recommendations, portfolio reviews, and market advisory discussions across equity, forex, commodity, crypto, and international stock markets. Each broker manages hundreds of clients and is expected to stay updated with overnight market developments before the market opens.

Every morning, brokers must review large volumes of financial news, stock-price movements, earnings discussions, analyst commentary, and SEC regulatory filings within a very limited preparation window. In practice, brokers can only read a small portion of the available information before their first client calls begin. As a result, many important signals, disclosures, and market events remain unnoticed.

This creates an intelligence gap during client interactions. Brokers often rely on partial information, memory, or fragmented market sources while answering client questions. Important disclosures buried inside long 10-K and 10-Q filings are difficult to review manually, and brokers may struggle to provide evidence-backed responses when clients ask for justification or supporting references. This increases both compliance risk and trust-related challenges.

To solve this problem, GlobalEdge wants to build a financial intelligence assistant that allows brokers to ask natural-language questions and receive grounded answers backed by real financial data, news articles, and regulatory filings. The system should reduce information overload, improve market coverage, and help brokers make faster and more confident advisory decisions without adding technical complexity to their workflow.

### Objective

This project proposes a proof-of-concept Retrieval-Augmented Generation (RAG) financial intelligence system that combines financial news, stock-price data, and SEC filings into a searchable intelligence layer. The system uses semantic retrieval with a local Chroma vector database to fetch relevant financial context and generate grounded answers for broker questions using large language models. Brokers can ask plain-English questions such as market sentiment analysis, company-risk queries, filing-related questions, or cross-market comparisons and receive evidence-backed responses.

To improve the quality and reliability of generated answers, the system introduces a DeepEval-based prompt optimization workflow using GEPA (Genetic-Pareto Prompt Optimization). Instead of manually refining prompts, the workflow uses benchmark financial question-answer examples and evaluation metrics such as faithfulness, groundedness, relevance, and actionability to optimize the final answering prompt.

The project also evaluates multiple RAG configurations by tuning retrieval and generation parameters such as chunk size, chunk overlap, number of retrieved chunks, temperature, top-p, and maximum token limits. Each configuration is evaluated using DeepEval metrics to identify the most effective combination for grounded financial reasoning and broker-style question answering.

The final system uses the optimized prompt together with the best-performing RAG configuration to generate accurate, grounded, and production-style financial intelligence responses for broker workflows.

### Data Description

The system uses three financial intelligence datasets collected through a custom data ingestion pipeline and stored locally for the RAG workflow.
1. Global Financial News (global_news.csv)
Contains financial news articles with titles, article content, publication dates, URLs, and source metadata. The dataset covers company announcements, market developments, macroeconomic events, sector movements, and sentiment-related signals.
2. Global Stock Prices (all_prices_clean.csv)
Contains historical stock-price data across global equity markets, crypto markets, and forex markets. Each record includes ticker symbols, company names, open/high/low/close prices, trading volume, and timestamps for market analysis and trend evaluation.
3. SEC Regulatory Filings (sec_filings.txt)
Contains regulatory filing documents including 10-K annual reports, 10-Q quarterly reports, and other SEC disclosures. These filings include financial statements, governance information, risk disclosures, operational commentary, and compliance-related information used for grounded financial reasoning.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installing the necessary libraries with specified versions
%pip install -qU \
    "chromadb==1.5.9" \
    "langchain-community==0.4.1" \
    "langchain-chroma==1.1.0" \
    "langchain-openai==1.2.1" \
    "langchain-text-splitters==1.1.2" \
    "pandas>=2.2.0" \
    "numpy>=1.26.0" \
    "pypdf>=5.0.0" \
    "scikit-learn>=1.4.0" \
    "python-dotenv>=1.0.1" \
    "tqdm>=4.66.0"\
    "deepeval"

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Importing the necessary libraries

# Standard Python libraries for file handling, text processing, JSON handling,
# randomness, timing, and basic utilities.
import os
import re
import json
import random
import time
from pathlib import Path
from collections import Counter
import pandas as pd
from dotenv import load_dotenv

# DeepEval libraries for prompt optimization and evaluation.
from deepeval.prompt import Prompt
from deepeval.dataset import Golden
from deepeval.metrics import GEval
from deepeval.optimizer import PromptOptimizer
from deepeval.optimizer.algorithms import GEPA
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import GPTModel
from deepeval.optimizer.policies import TieBreaker

# LangChain libraries for document loading, text splitting, embeddings,
# vector storage, and LLM interaction.
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

## Environment Setup and Project Initialization

In [ ]:
env_path = Path("../.env") if Path("../.env").exists() else Path(".env")
load_dotenv(env_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not found. Add it to your .env file before running the notebook.")

print(f"Loaded environment variables from: {env_path.resolve()}")

# Section 1: Baseline Retrieval-Augmented Generation (RAG) Pipeline

## 1.1 Load the CSV files with `CSVLoader`

In [ ]:
data_dir = Path("../data/raw") if Path("../data/raw").exists() else Path("data/raw")

stock_prices_path = data_dir / "stock_price_details.csv"
global_news_path = data_dir / "global_news.csv"

stock_loader = CSVLoader(file_path=str(stock_prices_path))
news_loader = CSVLoader(
    file_path=str(global_news_path),
    source_column="url",
    metadata_columns=["title", "source", "url", "published_at", "query"],
)

stock_docs = stock_loader.load()
news_docs = news_loader.load()

for doc in stock_docs:
    doc.metadata["dataset"] = "stock_price_details"

for doc in news_docs:
    doc.metadata["dataset"] = "global_news"

csv_docs = stock_docs + news_docs

print(f"Loaded {len(stock_docs)} stock price documents")
print(f"Loaded {len(news_docs)} news documents")
print(f"Loaded {len(csv_docs)} total CSV documents")
print(csv_docs[0].page_content[:500])

## 1.2 Load the SEC filings text and split it into chunks


In [ ]:
sec_pdf_path = data_dir / "sec_filings_10q.pdf"

if not sec_pdf_path.exists():
    raise FileNotFoundError("Expected sec_filings_10q.pdf in the raw data folder.")

from langchain_community.document_loaders import PyPDFLoader

sec_loader = PyPDFLoader(str(sec_pdf_path))

sec_raw_docs = sec_loader.load()

for doc in sec_raw_docs:
    doc.metadata["dataset"] = "sec_filings"

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

sec_docs = text_splitter.split_documents(sec_raw_docs)
all_docs = csv_docs + sec_docs

print(f"Loaded {len(sec_raw_docs)} SEC filing source documents")
print(f"Created {len(sec_docs)} SEC filing chunks")
print(f"Prepared {len(all_docs)} total documents for vector indexing")
print(sec_docs[0].page_content[:500])

## 1.3 Build a local Chroma vector store

In [ ]:
chroma_dir = Path("../chroma_db") if Path("../data/raw").exists() else Path("chroma_db")
collection_name = "investment_rag"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory=str(chroma_dir),
)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

print(f"Created Chroma collection: {collection_name}")
print(f"Persisted vector store at: {chroma_dir.resolve()}")
print(f"Indexed {vector_store._collection.count()} documents")

## 1.4 Retrieve Context and Generate Answers using the Baseline RAG Pipeline

In [ ]:
baseline_system_prompt = """
You are a grounded investment intelligence assistant for brokerage analysts.
Use only the retrieved context to answer the user's question.
If the context is insufficient, say what is missing instead of guessing.
Keep the answer concise, evidence-backed, and suitable for a broker speaking with a client.
""".strip()


def format_source_metadata(metadata):
    source_file = metadata.get("source_file") or metadata.get("source") or metadata.get("dataset", "unknown")
    title = metadata.get("title")
    published_at = metadata.get("published_at")
    page = metadata.get("page")
    row = metadata.get("row")

    details = [str(source_file)]
    if title:
        details.append(str(title))
    if published_at:
        details.append(str(published_at))
    if page is not None:
        details.append(f"page {page}")
    if row is not None:
        details.append(f"row {row}")

    return " | ".join(details)


def format_retrieved_context(docs):
    formatted_sources = []

    for idx, doc in enumerate(docs, start=1):
        metadata = doc.metadata or {}
        source_label = format_source_metadata(metadata)
        content = re.sub(r"\s+", " ", doc.page_content).strip()
        formatted_sources.append(f"[Source {idx}] {content}\nMetadata: {source_label}")

    return "\n\n".join(formatted_sources)


def retrieve_context(question, k=5):
    active_retriever = retriever
    if k != retriever.search_kwargs.get("k", 5):
        active_retriever = vector_store.as_retriever(search_kwargs={"k": k})

    return active_retriever.invoke(question)


def baseline_rag_answer(question, k=5):
    retrieved_docs = retrieve_context(question, k=k)
    context = format_retrieved_context(retrieved_docs)

    user_prompt = f"""
Question:
{question}

Retrieved context:
{context}

Answer with a direct response followed by a short Sources section naming the most relevant source numbers.
""".strip()

    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    response = llm.invoke([
        SystemMessage(content=baseline_system_prompt),
        HumanMessage(content=user_prompt),
    ])

    return {
        "question": question,
        "answer": response.content,
        "context": context,
        "source_documents": retrieved_docs,
    }


sample_question = "What are the main risks or signals a broker should discuss with a client considering Apple?"
baseline_sample = baseline_rag_answer(sample_question)

print("Question:")
print(baseline_sample["question"])
print("\nAnswer:")
print(baseline_sample["answer"])
print("\nRetrieved sources:")
for idx, doc in enumerate(baseline_sample["source_documents"], start=1):
    print(f"[{idx}] {format_source_metadata(doc.metadata or {})}")

## 1.5 Load the gold benchmark dataset and evaluate the baseline prompt


In [ ]:
# Write your code here

## 1.6 Define the evaluation metrics

In [ ]:
# Write your code here

## 1.7 Evaluate the baseline RAG on the golden examples dataset

In [ ]:
# Write your code here

# Section 2: DeepEval Prompt Optimization Layer

## 2.1 Define the prompt template and model callback


In [ ]:
# Write your code here

## 2.2 Optimize the prompt with GEPA

In [ ]:
# Write your code here

## 2.3 Evaluate the optimized prompt on the benchmark dataset

In [ ]:
# Write your code here

## 2.4 Compare optimized prompt with the baseline prompt


In [ ]:
# Write your code here

# Section 3: RAG Configuration Tuning and Evaluation


## 3.1 Define the RAG configurations

In [ ]:
# Write your code here

## 3.2 Create a runtime answering function

In [ ]:
# Write your code here

## 3.3 Evaluate one configuration on a benchmark set

In [ ]:
# Write your code here

## 3.4 Run all configurations on the test set

In [ ]:
# Write your code here

## 3.5 Compare configurations

In [ ]:
# Write your code here

## 3.6 Select the best overall approach

In [ ]:
# Write your code here

## 3.7 Save the tuning results


In [ ]:
# Write your code here

# Section 4: Final RAG Inference Pipeline with the Optimized Prompt and Best RAG Configuration

## 4.1 Define the final inference helper

In [ ]:
# Write your code here

## 4.2 Define the final test cases

In [ ]:
final_test_cases = [
    {
        "id": 1,
        "question": (
            "Compare the total revenue and net income reported by Apple, Amazon, "
            "and Alphabet in their latest 10-Q filings. Which company grew fastest "
            "year-over-year?"
        )
    },
    {
        "id": 2,
        "question": (
            "Based on this week's news sentiment around Amazon, would you classify "
            "the current signal as bullish or bearish, and does the price data support that?"
        )
    },
    {
        "id": 3,
        "question": (
            "Should I invest in Apple right now? I heard Tim Cook is stepping down — "
            "is that a red flag? How's the stock been doing, and is there anything I should "
            "be worried about with the company's financials or risks?"
        )
    },
    {
        "id": 4,
        "question": (
            "Is now a good time to buy Bitcoin? News is saying it just crossed $78k — "
            "am I too late? What's the price trend looked like over the last few months, "
            "and what's driving the current rally?"
        )
    },
    {
        "id": 5,
        "question": (
            "I want to invest in AI. Out of Google, Amazon, and Apple, which one looks "
            "like the best bet right now? How are they performing, what are they saying "
            "about their AI business, and what's the buzz around them?"
        )
    },
]

## 4.3 Run the final test cases

In [ ]:
# Write your code here

# Conclusion

## Actionable Insights:

- WRITE INSIGHTS HERE

## Recommendations:

- WRITE RECOMMENDATIONS HERE

<font size=6>Power Ahead!</font>
___